# Fine-tune Thai transformers for YouTube virality prediction

Run this notebook on **Google Colab (T4 / A100)** or **Kaggle**.  
It clones the repo, installs deps, and runs the heavy fine-tunes.

## Steps
1. Mount Google Drive (optional) for persistence
2. Clone the repo
3. Install Python deps (`uv` for speed)
4. Upload the processed dataset (or download from your bucket)
5. Run the fine-tune scripts


In [ ]:
# Verify GPU
!nvidia-smi

In [ ]:
# Clone the repo (replace with your fork if needed)
!git clone https://github.com/MNupakorn/thesis-youtube-virality-thai.git
%cd thesis-youtube-virality-thai

In [ ]:
# Install deps (CUDA wheels via pip, fast)
!pip install -q -U uv
!uv pip install --system -e ".[dev,gpu]"

In [ ]:
# Upload the processed dataset produced by `scripts/prepare_data.py` on your local machine
from google.colab import files
import os
os.makedirs('data/processed', exist_ok=True)
uploaded = files.upload()  # select dataset_with_labels.parquet

In [ ]:
# Hugging Face login (needed for Typhoon / OpenThaiGPT gated models)
from huggingface_hub import login
login()  # paste your HF token

In [ ]:
# 1) WangchanBERTa  (full fine-tune)
!python scripts/train_transformer.py --model wangchanberta --config configs/train.yaml

In [ ]:
# 2) Typhoon 2.5  (QLoRA)
!python scripts/train_transformer.py --model typhoon-2.5 --config configs/train.yaml

In [ ]:
# 3) OpenThaiGPT  (QLoRA — use 7B unless you have A100 80GB)
!python scripts/train_transformer.py --model openthaigpt --config configs/train.yaml

In [ ]:
# Zip predictions & metrics, download back to local
!tar czf artifacts.tgz reports/ mlruns/
from google.colab import files
files.download('artifacts.tgz')